# Geomagnetic Storm Prediction - Classification (PBL-I)
**Author:** Hrijit

This notebook handles the classification task for predicting geomagnetic storms.
We convert the continuous `dst` values into a binary class: Storm (1) for $Dst \le -50$, and Normal (0) for $Dst > -50$.

**Mitigation of Validation Set Anomaly:**
The original chronological split produced a validation set with 0 storm events. To properly validate our model without data leakage on the test set, we combined the Train and Validation sets and created a new chronological split (85/15) that ensures the validation set contains storms.

**Time-Series Post-Processing:**
Since geomagnetic storms are continuous events, we apply a 3-hour rolling average on the predicted probabilities to prevent "flickering" predictions.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../../results/classification', exist_ok=True)

# 1. Load Data
print("Loading data...")
train_raw = pd.read_csv('../../data/processed/train_final.csv')
val_raw = pd.read_csv('../../data/processed/validation_final.csv')
test_df = pd.read_csv('../../data/processed/test_final.csv')

# Combine train and val to resplit properly
combined = pd.concat([train_raw, val_raw], ignore_index=True)

# Find a good split point
split_idx = int(len(combined) * 0.85)
while True:
    val_df = combined.iloc[split_idx:]
    if (val_df['dst'] <= -50).sum() > 50:
        break
    split_idx -= 500
    if split_idx <= 0:
        break

train_df = combined.iloc[:split_idx].copy()
val_df = combined.iloc[split_idx:].copy()

print(f"Train storms: {(train_df['dst'] <= -50).sum()} / {len(train_df)}")
print(f"Val storms: {(val_df['dst'] <= -50).sum()} / {len(val_df)}")
print(f"Test storms: {(test_df['dst'] <= -50).sum()} / {len(test_df)}")

target_col = 'dst'
features = pd.read_csv('../../data/processed/feature_columns.csv')['feature'].tolist()
features = [f for f in features if f in train_df.columns]

X_train = train_df[features]
y_train = (train_df[target_col] <= -50).astype(int)

X_val = val_df[features]
y_val = (val_df[target_col] <= -50).astype(int)

X_test = test_df[features]
y_test = (test_df[target_col] <= -50).astype(int)


In [ ]:
# 2. Train Model
print("Training model...")
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
model = xgb.XGBClassifier(n_estimators=100, max_depth=5, scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)


In [ ]:
# 3. Predict and Smooth (Time-Series correction)
def predict_and_smooth(X, window=3):
    probas = model.predict_proba(X)[:, 1]
    # Smooth probabilities with rolling average
    smoothed_probas = pd.Series(probas).rolling(window=window, min_periods=1, center=True).mean().values
    preds = (smoothed_probas >= 0.5).astype(int)
    return probas, smoothed_probas, preds

train_probas, train_smooth, train_preds = predict_and_smooth(X_train)
val_probas, val_smooth, val_preds = predict_and_smooth(X_val)
test_probas, test_smooth, test_preds = predict_and_smooth(X_test)


In [ ]:
# 4. Evaluation and saving metrics
print("Evaluating...")
def save_metrics(y_true, y_pred, filepath):
    report = classification_report(y_true, y_pred, output_dict=True)
    df = pd.DataFrame(report).transpose()
    df.to_csv(filepath)
    return df

val_metrics = save_metrics(y_val, val_preds, '../../results/classification/classification_metrics.csv')
test_metrics = save_metrics(y_test, test_preds, '../../results/classification/test_metrics.csv')

print("Validation Metrics:")
display(val_metrics)
print("Test Metrics:")
display(test_metrics)


In [ ]:
# 5. Save Predictions for Pratistha
test_out = test_df[['period', 'hour', 'dst']].copy()
test_out['storm_pred'] = test_preds
test_out['storm_proba'] = test_smooth
test_out.to_csv('../../results/classification/classification_predictions.csv', index=False)
print("Predictions saved successfully.")


In [ ]:
# 6. Plot Confusion Matrices
def plot_cm(y_true, y_pred, filepath, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig(filepath)
    plt.show()

plot_cm(y_val, val_preds, '../../results/classification/confusion_matrix.png', 'Validation Confusion Matrix')
plot_cm(y_test, test_preds, '../../results/classification/test_confusion_matrix.png', 'Test Confusion Matrix')


In [ ]:
# 7. Feature Importance
plt.figure(figsize=(10,6))
feat_imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False).head(10)
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.savefig('../../results/classification/feature_importance.png')
plt.show()
